In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chương 4: ĐO LƯỜNG RỦI RO DANH MỤC (PHƯƠNG PHÁP TOÁN HỌC)


In [5]:
# 1. Cài đặt môi trường và đường dẫn
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# SỬA LẠI ĐƯỜNG DẪN TẠI ĐÂY: Trỏ về thư mục ../data/
DATA_PATH = '../data/processed_portfolio_2023_2024.csv'

In [6]:
# 2. Đọc dữ liệu
returns = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
assets = ['BTC_Return', 'GLD_Return', 'SPY_Return']
returns = returns[assets]

In [7]:
# 3. Tính Rủi ro từng tài sản riêng lẻ (Mục 4.1)
TRADING_DAYS = 252
asset_var = returns.var()
asset_std = returns.std()
asset_std_annual = asset_std * np.sqrt(TRADING_DAYS)

asset_risk_df = pd.DataFrame({
    'Variance (Daily)': asset_var,
    'Std Dev (Daily %)': asset_std * 100,
    'Annualized Std (%)': asset_std_annual * 100
})
asset_risk_df.index = ['BTC', 'GLD', 'SPY']
asset_risk_df.to_csv('../outputs/asset_variance_std.csv')
print("--- Rủi ro tài sản riêng lẻ ---")
print(asset_risk_df)

--- Rủi ro tài sản riêng lẻ ---
     Variance (Daily)  Std Dev (Daily %)  Annualized Std (%)
BTC          0.001049           3.238507           51.409703
GLD          0.000080           0.896188           14.226537
SPY          0.000065           0.807658           12.821181


In [8]:
# 4. Ma trận Hiệp phương sai (Mục 4.2)
cov_matrix = returns.cov()
cov_matrix.index = ['BTC', 'GLD', 'SPY']
cov_matrix.columns = ['BTC', 'GLD', 'SPY']
cov_matrix.to_csv('../outputs/covariance_matrix.csv')

# Vẽ Heatmap Covariance
plt.figure(figsize=(8, 6))
sns.heatmap(cov_matrix, annot=True, fmt=".8f", cmap="YlGnBu", cbar_kws={'shrink': .8})
plt.title('Ma tran Hiep phuong sai (Covariance Matrix)')
plt.tight_layout()
plt.savefig('../figures/covariance_heatmap.png', dpi=300)
plt.close()

In [9]:
# 5. Rủi ro danh mục - Tổ hợp tuyến tính (Mục 4.3)
# Tỷ trọng Equal Weight
w = np.array([1/3, 1/3, 1/3])

# Tính Portfolio Variance: w.T * Cov * w
port_var = w.T @ cov_matrix.values @ w

# Tính Portfolio Standard Deviation (Volatility)
port_std = np.sqrt(port_var)
port_std_annual = port_std * np.sqrt(TRADING_DAYS)

port_risk_df = pd.DataFrame({
    'Metric': ['Portfolio Variance (Daily)', 'Portfolio Volatility (Daily %)', 'Annualized Volatility (%)'],
    'Value': [port_var, port_std * 100, port_std_annual * 100]
})
port_risk_df.to_csv('../outputs/portfolio_risk.csv', index=False)

print("\n--- Rủi ro Danh mục (Equal Weight) ---")
print(port_risk_df)


--- Rủi ro Danh mục (Equal Weight) ---
                           Metric      Value
0      Portfolio Variance (Daily)   0.000156
1  Portfolio Volatility (Daily %)   1.250379
2       Annualized Volatility (%)  19.849157
